# core

> litert model initialisation and tools

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, re
from html import escape
from mimetypes import guess_type
from litert_lm import (Engine, Backend, Conversation, Session, Message, Contents,
                       Content, Role, ToolCall, ToolEventHandler, set_min_log_severity)
from litert_lm._messages import (Text, ImageBytes, ImageFile, AudioBytes, AudioFile,
                                 ToolResponse, normalize_message)
from huggingface_hub import hf_hub_download
from fastcore.all import Path, store_attr, patch, L, GetAttr, ifnone
from safepyrun import RunPython

In [ ]:
#| export
def mk_content(o):
    "Convert `o` to a litert `Content`."
    if isinstance(o, Content): return o
    if isinstance(o, str): return Text(o)
    if isinstance(o, bytes): return ImageBytes(o)
    if isinstance(o, Path):
        mime = guess_type(str(o))[0] or ''
        return AudioFile(str(o)) if mime.startswith('audio/') else ImageFile(str(o))
    raise TypeError(f"Unsupported content type: {type(o)}")

def mk_msg(content, role='user'):
    "Create a litert `Message` from str/bytes/list/dict/Message."
    if content is None: return None
    if isinstance(content, Message): return content
    if isinstance(content, dict): return Message(Role(content['role']), Contents.of(content['content']))
    parts = [mk_content(o) for o in content] if isinstance(content, list) else [mk_content(content)]
    return Message(Role(role), Contents(parts))

def mk_msgs(msgs):
    "Normalize a list of messages to litert message dicts."
    if not msgs: return []
    if not isinstance(msgs, list): msgs = [msgs]
    return [normalize_message(m if isinstance(m, (Message, dict)) else mk_msg(m)) for m in msgs]

In [ ]:
#| hide
from fastcore.test import test_eq
test_eq(mk_msg("hello").to_json(), {"role": "user", "content": [{"type": "text", "text": "hello"}]})
test_eq(mk_msg("hi", role="model").to_json()["role"], "model")
test_eq([x["role"] for x in mk_msgs(["a", mk_msg("b", role="model")])], ["user", "model"])
assert isinstance(mk_content("x"), Text)

litertlm defaults.

In [ ]:
#| export
gemma4_e4b='litert-community/gemma-4-E4B-it-litert-lm'
gemma4_e2b='litert-community/gemma-4-E2B-it-litert-lm'
gemma4_12b='litert-community/gemma-4-12B-it-litert-lm'

In [ ]:
#| export

def get_model(model_id: str, model_path:str):
	if model_path and Path(model_path).exists(): return model_path
	return hf_hub_download(model_id, repo_type='model')

class Chat:
	def __init__(self,
        model_id:str=gemma4_e2b, # hf model id
        model_path:str=None, # local model path
        backend:Backend=Backend.CPU, # litert backend
        multimodal:bool=True,
        sp:str='',
        messages:list=None,
		tools:list=None,


        **kwargs
	):
		"Local Gemma engine over litert_lm; downloads the model on first use."
		assert model_path or model_id, "model_id or model_path must be provided"
		model_path = model_path or hf_hub_download(model_id, repo_type='model')
		self.engine = Engine(model_path, backend=backend, multimodal=multimodal)
		self.conv: Conversation|None = None





In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()